<div style="background: linear-gradient(135deg, #4a148c 0%, #880e4f 100%); padding: 30px;
            border-radius: 15px; color: white; text-align: center;">
  <h1 style="font-size: 2.2em; margin-bottom: 8px;">📊 DASHBOARD KẾT QUẢ THỰC NGHIỆM</h1>
  <h2 style="font-weight: normal; font-size: 1.2em;">Banknote Authentication — Tổng hợp & Trực quan hóa</h2>
  <hr style="border: 1px solid rgba(255,255,255,0.3); margin: 15px 0;">
  <p>📁 File này tổng hợp toàn bộ kết quả từ <b>4 lần chạy thực nghiệm</b></p>
  <p>Chạy từng cell theo thứ tự để tái tạo tất cả biểu đồ & bảng phân tích.</p>
</div>

In [ ]:
# ── Setup ──────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display, HTML, Image

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': '#fafafa'})

# ── Dữ liệu kết quả thực tế ────────────────────────────────────
baseline_data = {
    "Model":         ["kNN","Naive Bayes","SVM","Decision Tree","Random Forest",
                      "AdaBoost","Gradient Boosting","LDA","MLP","Logistic Regression"],
    "CV Acc (mean)": [0.9969, 0.8396, 0.9990, 0.9844, 0.9896, 0.9938, 0.9896, 0.9760, 1.0000, 0.9833],
    "CV Acc (std)" : [0.0026, 0.0101, 0.0021, 0.0114, 0.0099, 0.0039, 0.0109, 0.0053, 0.0000, 0.0083],
    "Accuracy"     : [1.0000, 0.8641, 1.0000, 0.9879, 0.9951, 0.9976, 0.9927, 0.9684, 1.0000, 0.9782],
    "Precision"    : [1.0000, 0.8547, 1.0000, 0.9785, 0.9892, 0.9946, 0.9839, 0.9337, 1.0000, 0.9531],
    "Recall"       : [1.0000, 0.8361, 1.0000, 0.9945, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
    "F1-Score"     : [1.0000, 0.8453, 1.0000, 0.9864, 0.9946, 0.9973, 0.9919, 0.9657, 1.0000, 0.9760],
    "AUC"          : [1.0000, 0.9483, 1.0000, 0.9885, 0.9999, 1.0000, 0.9998, 0.9998, 1.0000, 0.9999],
    "Time (s)"     : [0.023,  0.013,  0.110,  0.017,  0.871,  0.513,  0.851,  0.018,  2.257,  0.024],
    "Type"         : ["Baseline"] * 10
}
tuned_data = {
    "Model"     : ["RF (Tuned)","SVM (Tuned)","GB (Tuned)","kNN (Tuned)","MLP (Tuned)","Stacking Ensemble"],
    "Accuracy"  : [0.9951, 1.0000, 0.9976, 1.0000, 1.0000, 1.0000],
    "Precision" : [0.9892, 1.0000, 0.9946, 1.0000, 1.0000, 1.0000],
    "Recall"    : [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
    "F1-Score"  : [0.9946, 1.0000, 0.9973, 1.0000, 1.0000, 1.0000],
    "AUC"       : [0.9999, 1.0000, 0.9999, 1.0000, 1.0000, 1.0000],
    "Type"      : ["Tuned", "Tuned", "Tuned", "Tuned", "Tuned", "Stacking"]
}

df_base  = pd.DataFrame(baseline_data).set_index("Model")
df_tuned = pd.DataFrame(tuned_data).set_index("Model")
print("✅ Dữ liệu thực nghiệm đã được tải vào bộ nhớ.")

---
## 📊 CHART 1 — Bảng tổng kết kết quả

In [ ]:
display(HTML("<h3>📋 Bảng 1: Kết quả 10 Thuật toán Baseline</h3>"))

def color_row(val):
    if isinstance(val, float):
        if val == 1.0:   return 'background-color: #c8e6c9; font-weight: bold'
        elif val >= 0.99: return 'background-color: #dcedc8'
        elif val < 0.90:  return 'background-color: #ffcdd2'
    return ''

cols_show = ["CV Acc (mean)", "CV Acc (std)", "Accuracy", "Precision", "Recall", "F1-Score", "AUC", "Time (s)"]
styled = df_base[cols_show].style\
    .applymap(color_row)\
    .format("{:.4f}", subset=["CV Acc (mean)","CV Acc (std)","Accuracy","Precision","Recall","F1-Score","AUC"])\
    .format("{:.3f}s", subset=["Time (s)"])\
    .set_caption("🟢 Xanh lá = F1≥0.99   🔴 Đỏ nhạt = F1<0.90")
display(styled)

display(HTML("<h3>📋 Bảng 2: Kết quả Tuning & Stacking Ensemble</h3>"))
cols_t = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]
styled2 = df_tuned[cols_t].style\
    .applymap(color_row)\
    .format("{:.4f}")\
    .set_caption("🟢 = Hoàn hảo (1.0000)")
display(styled2)

---
## 📈 CHART 2 — So sánh F1-Score toàn bộ mô hình

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle("So sánh F1-Score — Baseline vs Tuned/Stacking",
             fontsize=15, fontweight="bold", y=1.01)

# -- Subplot 1: Baseline
ax1 = axes[0]
b_sorted = df_base["F1-Score"].sort_values()
palette  = ["#e57373" if v < 0.90 else ("#66bb6a" if v == 1.0 else "#42a5f5") for v in b_sorted]
bars1    = ax1.barh(b_sorted.index, b_sorted.values, color=palette, edgecolor="white", height=0.6)
for bar, val in zip(bars1, b_sorted.values):
    ax1.text(val + 0.003, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=9, fontweight="bold")
ax1.set_xlim(b_sorted.min() - 0.04, 1.035)
ax1.set_title("Baseline (10 mô hình)", fontsize=12, fontweight="bold", color="#1a237e")
ax1.set_xlabel("F1-Score")
ax1.axvline(1.0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax1.add_patch(mpatches.Patch(color='#66bb6a', label='F1 = 1.00 (Hoàn hảo)'))
ax1.add_patch(mpatches.Patch(color='#42a5f5', label='F1 ≥ 0.90'))
ax1.add_patch(mpatches.Patch(color='#e57373', label='F1 < 0.90 (Yếu)'))
ax1.legend(fontsize=8, loc='lower right')

# -- Subplot 2: Tuned + Stacking
ax2 = axes[1]
t_sorted = df_tuned["F1-Score"].sort_values()
t_colors = ["#ab47bc" if "Stacking" in n else "#ffa726" for n in t_sorted.index]
bars2    = ax2.barh(t_sorted.index, t_sorted.values, color=t_colors, edgecolor="white", height=0.6)
for bar, val in zip(bars2, t_sorted.values):
    ax2.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=9, fontweight="bold")
ax2.set_xlim(t_sorted.min() - 0.006, 1.01)
ax2.set_title("Tuned + Stacking Ensemble", fontsize=12, fontweight="bold", color="#4a148c")
ax2.set_xlabel("F1-Score")
ax2.axvline(1.0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax2.add_patch(mpatches.Patch(color='#ffa726', label='Mô hình Tuned'))
ax2.add_patch(mpatches.Patch(color='#ab47bc', label='Stacking Ensemble'))
ax2.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig("dashboard_f1_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] dashboard_f1_comparison.png")

---
## 📈 CHART 3 — Radar Chart: So sánh đa chỉ số Top 5 mô hình

In [ ]:
metrics   = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]
top5      = ["kNN", "SVM", "MLP", "Random Forest", "Naive Bayes"]
N         = len(metrics)
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles   += angles[:1]
colors5   = ["#2196F3", "#4CAF50", "#9C27B0", "#FF9800", "#F44336"]

fig, axes = plt.subplots(1, 5, figsize=(20, 5), subplot_kw=dict(polar=True))
fig.suptitle("Radar Chart — So sánh đa chỉ số (Top 4 + Yếu nhất)",
             fontsize=14, fontweight="bold", y=1.04)

for ax, name, color in zip(axes, top5, colors5):
    values = [df_base.loc[name, m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=2, linestyle='solid')
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, size=8)
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_yticklabels(["0.6","0.7","0.8","0.9","1.0"], size=6)
    ax.set_title(name, size=10, fontweight="bold", pad=15, color=color)
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig("dashboard_radar.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] dashboard_radar.png")

---
## 📈 CHART 4 — Heatmap tổng hợp tất cả mô hình × chỉ số

In [ ]:
# Ghép baseline + tuned để vẽ heatmap toàn bộ
hm_base            = df_base[metrics].copy()
hm_tuned           = df_tuned[metrics].copy()
hm_tuned.index     = [f"{i}  ★" for i in hm_tuned.index]
hm_all             = pd.concat([hm_base, hm_tuned])
hm_all_sorted      = hm_all.sort_values("F1-Score")

fig, ax = plt.subplots(figsize=(9, 11))
sns.heatmap(
    hm_all_sorted,
    annot=True, fmt=".4f", annot_kws={"size": 9},
    cmap="RdYlGn", vmin=0.82, vmax=1.0,
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={"shrink": 0.7}
)
ax.set_title("Heatmap Tổng Hợp — Tất cả mô hình × Chỉ số\n(★ = Tuned/Stacking)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Chỉ số đánh giá", fontsize=11)
ax.set_ylabel("")
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig("dashboard_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] dashboard_heatmap.png")

---
## 📈 CHART 5 — Bubble Chart: Accuracy vs F1 vs Thời gian thực thi

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

bubble_colors = plt.cm.tab10(np.linspace(0, 1, len(df_base)))
time_vals     = df_base["Time (s)"].values
bubble_size   = (time_vals / time_vals.max()) * 2500 + 200

scatter = ax.scatter(
    df_base["F1-Score"],
    df_base["Accuracy"],
    s=bubble_size,
    c=bubble_colors,
    alpha=0.75,
    edgecolors='white',
    linewidths=1.5
)

for i, (name, row) in enumerate(df_base.iterrows()):
    offset_x = -0.042 if name in ["kNN", "SVM"] else 0.003
    offset_y = -0.01  if name == "MLP" else 0.004
    ax.annotate(name,
                xy=(row["F1-Score"], row["Accuracy"]),
                xytext=(row["F1-Score"] + offset_x, row["Accuracy"] + offset_y),
                fontsize=9, fontweight='bold', color=bubble_colors[i])

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.4)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.4)
ax.set_xlabel("F1-Score", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Bubble Chart: Accuracy vs F1-Score\n(Kích thước bong bóng = Thời gian huấn luyện)",
             fontsize=13, fontweight='bold')
ax.set_xlim(0.82, 1.025)
ax.set_ylim(0.84, 1.02)

# Legend kích thước
for t_val, label in [(0.1, "0.1s"), (0.5, "0.5s"), (2.0, "2.0s")]:
    sz = (t_val / time_vals.max()) * 2500 + 200
    ax.scatter([], [], s=sz, c='gray', alpha=0.5, label=label)
ax.legend(title="Thời gian train", loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig("dashboard_bubble.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] dashboard_bubble.png")

---
## 📈 CHART 6 — Biểu đồ cải thiện trước/sau Tuning

In [ ]:
mapping = {"RF (Tuned)": "Random Forest", "SVM (Tuned)": "SVM",
           "GB (Tuned)": "Gradient Boosting", "kNN (Tuned)": "kNN", "MLP (Tuned)": "MLP"}

names_display = list(mapping.keys())
base_f1s  = [df_base.loc[mapping[k], "F1-Score"] for k in names_display]
tuned_f1s = [df_tuned.loc[k, "F1-Score"]          for k in names_display]
deltas    = [t - b for t, b in zip(tuned_f1s, base_f1s)]

x     = np.arange(len(names_display))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle("Hiệu quả Hyperparameter Tuning", fontsize=14, fontweight='bold')

# -- Plot 1: Grouped bar before/after
bars_b = ax1.bar(x - width/2, base_f1s,  width, label='Baseline', color='#42a5f5', edgecolor='white', alpha=0.85)
bars_t = ax1.bar(x + width/2, tuned_f1s, width, label='Tuned',    color='#ff7043', edgecolor='white', alpha=0.85)

for bar, val in list(zip(bars_b, base_f1s)) + list(zip(bars_t, tuned_f1s)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f"{val:.4f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels([n.replace(" (Tuned)", "") for n in names_display], fontsize=10)
ax1.set_ylim(0.988, 1.006)
ax1.set_ylabel("F1-Score")
ax1.set_title("F1-Score: Baseline vs Tuned", fontweight='bold')
ax1.legend()
ax1.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.4)

# -- Plot 2: Delta (cải thiện)
delta_colors = ["#66bb6a" if d > 0 else "#ef5350" if d < 0 else "#bdbdbd" for d in deltas]
bars_d       = ax2.bar(x, deltas, color=delta_colors, edgecolor='white', alpha=0.85)
for bar, val in zip(bars_d, deltas):
    ax2.text(bar.get_x() + bar.get_width()/2, max(val, 0) + 0.00005,
             f"{val:+.4f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels([n.replace(" (Tuned)", "") for n in names_display], fontsize=10)
ax2.axhline(0, color='black', linewidth=1)
ax2.set_ylabel("ΔF1 (Tuned − Baseline)")
ax2.set_title("Mức cải thiện F1 sau Tuning", fontweight='bold')
ax2.add_patch(mpatches.Patch(color='#66bb6a', label='Cải thiện'))
ax2.add_patch(mpatches.Patch(color='#bdbdbd', label='Không đổi'))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("dashboard_delta.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Saved] dashboard_delta.png")

---
## 🏆 CHART 7 — Bảng Xếp hạng cuối cùng (Scoreboard)

In [ ]:
# Tạo scoreboard HTML đẹp
all_models = []

for name, row in df_base.iterrows():
    all_models.append({"Mô hình": name, "Loại": "Baseline",
                        "F1": row["F1-Score"], "AUC": row["AUC"],
                        "Accuracy": row["Accuracy"]})
for name, row in df_tuned.iterrows():
    all_models.append({"Mô hình": name, "Loại": row["Type"],
                        "F1": row["F1-Score"], "AUC": row["AUC"],
                        "Accuracy": row["Accuracy"]})

all_df = pd.DataFrame(all_models).sort_values("F1", ascending=False).reset_index(drop=True)

def make_medal(rank):
    return {0: "🥇", 1: "🥈", 2: "🥉"}.get(rank, f"#{rank+1}")

def make_badge(tag):
    if tag == "Stacking":  return "<span style='background:#ab47bc;color:white;padding:2px 7px;border-radius:10px;font-size:0.8em;'>Stacking</span>"
    if tag == "Tuned":     return "<span style='background:#ff7043;color:white;padding:2px 7px;border-radius:10px;font-size:0.8em;'>Tuned</span>"
    return "<span style='background:#42a5f5;color:white;padding:2px 7px;border-radius:10px;font-size:0.8em;'>Baseline</span>"

html = """
<style>
.scoreboard { border-collapse: collapse; width: 100%; font-family: 'Segoe UI', sans-serif; }
.scoreboard th { background: #4a148c; color: white; padding: 10px 14px; text-align: left; }
.scoreboard td { padding: 8px 14px; border-bottom: 1px solid #eee; }
.scoreboard tr:hover td { background: #f3e5f5; }
.perfect { background: #e8f5e9; font-weight: bold; }
.good    { background: #fff8e1; }
.weak    { background: #fce4ec; }
</style>
<table class='scoreboard'>
<tr><th>Rank</th><th>Mô hình</th><th>Loại</th><th>F1-Score</th><th>AUC</th><th>Accuracy</th></tr>
"""
for i, row in all_df.iterrows():
    cls = "perfect" if row["F1"] == 1.0 else ("good" if row["F1"] >= 0.99 else "weak")
    f1_bar = "▓" * int(row["F1"] * 10) + "░" * (10 - int(row["F1"] * 10))
    html += f"""<tr class='{cls}'>
        <td><b>{make_medal(i)}</b></td>
        <td><b>{row['Mô hình']}</b></td>
        <td>{make_badge(row['Loại'])}</td>
        <td>{row['F1']:.4f} <span style='color:#aaa;font-size:0.8em'>{f1_bar}</span></td>
        <td>{row['AUC']:.4f}</td>
        <td>{row['Accuracy']:.4f}</td>
    </tr>\n"""
html += "</table>"
display(HTML("<h3>🏆 Bảng Xếp Hạng Cuối Cùng — 16 Mô hình</h3>"))
display(HTML(html))

---
## 📝 TÓM TẮT CUỐI — Báo cáo tự động

In [ ]:
perfect_base    = (df_base["F1-Score"] == 1.0).sum()
good_base       = (df_base["F1-Score"] >= 0.99).sum()
weakest         = df_base["F1-Score"].idxmin()
fastest         = df_base["Time (s)"].idxmin()
slowest         = df_base["Time (s)"].idxmax()
perfect_tuned   = (df_tuned["F1-Score"] == 1.0).sum()

summary_html = f"""
<div style='background:linear-gradient(135deg,#0d47a1,#1565c0);color:white;padding:25px;
            border-radius:14px;font-family:Segoe UI,sans-serif;line-height:1.8;'>
  <h2 style='margin-top:0;border-bottom:1px solid rgba(255,255,255,0.3);padding-bottom:10px;'>
    📝 BÁO CÁO TỰ ĐỘNG — Banknote Authentication
  </h2>
  <div style='display:grid;grid-template-columns:1fr 1fr;gap:20px;'>
    <div style='background:rgba(255,255,255,0.12);border-radius:10px;padding:15px;'>
      <h3 style='margin-top:0;color:#90caf9;'>📦 Dataset</h3>
      <p>• <b>1372 mẫu</b>, 4 đặc trưng Wavelet, không missing<br>
         • Cân bằng: Genuine 55.5% / Forged 44.5%<br>
         • Train: 960 mẫu | Test: 412 mẫu<br>
         • Tiền xử lý: StandardScaler (fit on train only)</p>
    </div>
    <div style='background:rgba(255,255,255,0.12);border-radius:10px;padding:15px;'>
      <h3 style='margin-top:0;color:#a5d6a7;'>🚀 Lần chạy 1: Baseline</h3>
      <p>• <b>{perfect_base}/10</b> mô hình đạt F1 = 1.00<br>
         • <b>{good_base}/10</b> mô hình đạt F1 ≥ 0.99<br>
         • Yếu nhất: <b>{weakest}</b> (F1={df_base.loc[weakest,'F1-Score']:.4f})<br>
         • Nhanh nhất: <b>{fastest}</b> ({df_base.loc[fastest,'Time (s)']:.3f}s)<br>
         • Chậm nhất: <b>{slowest}</b> ({df_base.loc[slowest,'Time (s)']:.3f}s)</p>
    </div>
    <div style='background:rgba(255,255,255,0.12);border-radius:10px;padding:15px;'>
      <h3 style='margin-top:0;color:#ffcc80;'>🔧 Lần chạy 3: Tuning (GridSearchCV)</h3>
      <p>• <b>{perfect_tuned}/6</b> mô hình (kể cả Stacking) đạt F1 = 1.00<br>
         • GB: F1 0.9919 → 0.9973 (cải thiện +0.0054)<br>
         • RF: Không thay đổi (đã gần tối ưu từ baseline)<br>
         • SVM, kNN, MLP: Xác nhận F1 = 1.00 ✅</p>
    </div>
    <div style='background:rgba(255,255,255,0.12);border-radius:10px;padding:15px;'>
      <h3 style='margin-top:0;color:#ce93d8;'>🧩 Lần chạy 4: Stacking Ensemble</h3>
      <p>• Base: RF + SVM + GB + kNN (Tuned)<br>
         • Meta-learner: Logistic Regression<br>
         • CV: 5-fold out-of-fold predictions<br>
         • F1 = AUC = Accuracy = <b>1.0000</b> ✅</p>
    </div>
  </div>
  <div style='background:rgba(255,255,255,0.08);border-radius:10px;padding:15px;margin-top:15px;'>
    <h3 style='margin-top:0;color:#f48fb1;'>🎯 Kết luận & Hướng phát triển</h3>
    <p>
    Bài toán phân loại tiền giấy thật/giả <b>giải quyết hoàn toàn bằng ML cổ điển</b>.
    4 đặc trưng Wavelet đủ mạnh, không cần Deep Learning với dataset này.<br><br>
    <b>Khuyến nghị:</b> Nếu mở rộng quy mô → thử <i>Bayesian Optimization (Optuna)</i>,
    <i>Feature Selection (RFE/LASSO)</i>, hoặc <i>CNN trực tiếp từ ảnh tờ tiền</i>.
    </p>
  </div>
</div>
"""
display(HTML(summary_html))

In [ ]:
# ── Danh sách file dashboard đã tạo ─────────────────────────
import os
dashboard_files = [
    ("dashboard_f1_comparison.png", "So sánh F1 Baseline vs Tuned"),
    ("dashboard_radar.png",         "Radar chart đa chỉ số Top 5"),
    ("dashboard_heatmap.png",       "Heatmap toàn bộ mô hình × chỉ số"),
    ("dashboard_bubble.png",        "Bubble chart: Acc vs F1 vs Time"),
    ("dashboard_delta.png",         "Mức cải thiện sau Tuning"),
]
print("📁 DASHBOARD FILES ĐÃ TẠO:")
print("─" * 55)
for fname, desc in dashboard_files:
    exists = "✅" if os.path.exists(fname) else "⏳ (Chạy cell trên để tạo)"
    print(f"  {exists}  {fname:<36} — {desc}")